# Customer-Level Uplift Modeling

## Introduction

The previous notebooks answered two important questions:

1. Did the email campaigns improve outcomes?
2. Were the observed effects statistically reliable?

This notebook asks a more advanced business question:

> Which customers are most likely to benefit from receiving an email?

This is different from ordinary predictive modeling.

A normal conversion model predicts:

> Who is likely to purchase?

An uplift model predicts:

> Whose purchase probability changes because of treatment?

That distinction matters.

A customer who would purchase anyway may not need an email.

A customer who will never purchase may also not be worth targeting.

The most valuable customers are often those whose behavior changes because of the campaign.

## Workflow

Customer features  
→ Treatment assignment  
→ Separate treatment and control models  
→ Predict potential outcomes  
→ Estimate customer-level uplift  
→ Rank customers  
→ Evaluate targeting strategy  
→ Estimate business value

In [1]:
# Core data tools
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    log_loss
)

In [2]:
# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", "{:,.4f}".format)

In [4]:
# Load cleaned data
DATA_PATH = "../data/processed/hillstrom_cleaned.csv"

df = pd.read_csv(DATA_PATH)

df.head()

,recency,history_segment,history,mens,womens,zip_code,newbie,channel,segment,visit,conversion,spend
0,10,2) $100 - $200,142.4400,1,0,Surburban,0,Phone,Womens E-Mail,0,0,0.0000
1,6,3) $200 - $350,329.0800,1,1,Rural,1,Web,No E-Mail,0,0,0.0000
2,7,2) $100 - $200,180.6500,0,1,Surburban,1,Web,Womens E-Mail,0,0,0.0000
3,9,5) $500 - $750,675.8300,1,0,Rural,1,Web,Mens E-Mail,0,0,0.0000
4,2,1) $0 - $100,45.3400,1,0,Urban,0,Web,Womens E-Mail,0,0,0.0000


Validate the data

In [5]:
# Check dataset structure
print("Dataset shape:", df.shape)
print("\nMissing values:", df.isna().sum().sum())
print("\nExperiment groups:")
print(df["segment"].value_counts())

Dataset shape: (64000, 12)

Missing values: 0

Experiment groups:
segment
Womens E-Mail    21387
Mens E-Mail      21307
No E-Mail        21306
Name: count, dtype: int64


## Feature Selection and Leakage Prevention

Uplift modeling requires special care with feature selection.

We may only use information that was available **before treatment assignment**.

Therefore, we must exclude:

- `segment` — this defines treatment assignment
- `visit` — this occurs after treatment
- `conversion` — this is the outcome
- `spend` — this occurs after treatment

Using post-treatment variables would create leakage and invalidate the customer targeting model.

The model will use only pre-treatment customer characteristics.

In [6]:
# Features available before treatment
feature_columns = [
    "recency",
    "history_segment",
    "history",
    "mens",
    "womens",
    "zip_code",
    "newbie",
    "channel"
]

# Binary outcome
target_column = "conversion"

feature_columns

['recency',
 'history_segment',
 'history',
 'mens',
 'womens',
 'zip_code',
 'newbie',
 'channel']

Mens E-Mail
vs
No E-Mail

# Part 1: Mens E-Mail Uplift Model

We first estimate heterogeneous treatment effects for:

- Treatment: Mens E-Mail
- Control: No E-Mail

The Womens E-Mail group is temporarily excluded from this comparison.

This gives us a clean binary treatment problem.

In [7]:
# Keep only Mens E-Mail and No E-Mail groups
mens_df = df[
    df["segment"].isin([
        "Mens E-Mail",
        "No E-Mail"
    ])
].copy()

# Create binary treatment indicator
mens_df["treatment"] = (
    mens_df["segment"] == "Mens E-Mail"
).astype(int)

# Check treatment distribution
mens_df["treatment"].value_counts()

treatment
1    21307
0    21306
Name: count, dtype: int64

Build modeling dataset

In [8]:
# Customer features
X = mens_df[feature_columns].copy()

# Conversion outcome
y = mens_df[target_column].copy()

# Treatment assignment
treatment = mens_df["treatment"].copy()

print("X shape:", X.shape)
print("Outcome rate:", y.mean())
print("Treatment rate:", treatment.mean())

X shape: (42613, 8)
Outcome rate: 0.009128669654800178
Treatment rate: 0.5000117335085538


In [12]:
X

,recency,history_segment,history,mens,womens,zip_code,newbie,channel
1,6,3) $200 - $350,329.0800,1,1,Rural,1,Web
3,9,5) $500 - $750,675.8300,1,0,Rural,1,Web
8,9,5) $500 - $750,675.0700,1,1,Rural,1,Phone
13,2,2) $100 - $200,101.6400,0,1,Urban,0,Web
14,4,3) $200 - $350,241.4200,0,1,Rural,1,Multichannel
...,...,...,...,...,...,...,...,...
63994,7,1) $0 - $100,86.4600,0,1,Urban,0,Web
63995,10,2) $100 - $200,105.5400,1,0,Urban,0,Web
63996,5,1) $0 - $100,38.9100,0,1,Urban,1,Phone
63997,6,1) $0 - $100,29.9900,1,0,Urban,1,Phone


In [13]:
y

1        0
3        0
8        0
13       0
14       0
        ..
63994    0
63995    0
63996    0
63997    0
63999    0
Name: conversion, Length: 42613, dtype: int64

In [14]:
treatment

1        0
3        1
8        1
13       1
14       0
        ..
63994    1
63995    1
63996    1
63997    1
63999    1
Name: treatment, Length: 42613, dtype: int64

Split correctly

In [9]:
# Combine treatment and outcome for stratification
stratify_key = (
    treatment.astype(str)
    + "_"
    + y.astype(str)
)

In [10]:
# Split customer indices
train_idx, test_idx = train_test_split(
    np.arange(len(X)),
    test_size=0.25,
    random_state=42,
    stratify=stratify_key
)


In [11]:
train_idx

array([21103, 21758,   364, ...,  5398, 26956, 35891], shape=(31959,))

In [15]:
# Create train data
X_train = X.iloc[train_idx].copy()
y_train = y.iloc[train_idx].copy()
treatment_train = treatment.iloc[train_idx].copy()

In [16]:
# Create test data
X_test = X.iloc[test_idx].copy()
y_test = y.iloc[test_idx].copy()
treatment_test = treatment.iloc[test_idx].copy()

print("Train size:", len(X_train))
print("Test size:", len(X_test))

Train size: 31959
Test size: 10654


Define feature Types

In [17]:
# Numerical features
numerical_features = [
    "recency",
    "history"
]

# Categorical features
categorical_features = [
    "history_segment",
    "mens",
    "womens",
    "zip_code",
    "newbie",
    "channel"
]

Build Preprocessing

In [18]:
# Numerical preprocessing
numerical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

# Categorical preprocessing
categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

# Combine preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numerical_pipeline,
            numerical_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ]
)